# LLM Agent를 위한 메모리 시스템(Agent Memory)

이 노트북은 스터디 코스에 memory system 주제를 추가한다. 목표는 memory를 추상 개념으로 두지 않고, 무엇을 저장하고 어떻게 검색하며 언제 업데이트해야 하는지, 그리고 careless한 memory 설계가 왜 좋은 agent도 망칠 수 있는지 구체적으로 보는 것이다.

## 학습 목표

- 단일 context window를 넘어 agent 시스템에 memory가 왜 필요한지 이해한다.
- short-term, long-term, episodic, semantic, vector memory 패턴을 구분한다.
- memory retrieval과 memory update 전략을 직접 실험한다.
- notebook 안에 로직을 숨기지 않고, 기존 workflow에 memory를 어떻게 연결할 수 있는지 확인한다.


## 개념 설명

다른 노트북과 마찬가지로 먼저 현재 Python 실행 환경을 확인한다. 이렇게 해야 Jupyter가 임의의 시스템 interpreter가 아니라 `uv`가 관리하는 kernel을 쓰고 있는지 바로 알 수 있다.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(sys.executable)

이 setup 셀은 작업 디렉터리를 안정적으로 맞추고, `src/`에 있는 memory class와 helper를 불러온다. 노트북은 교육용으로 읽히되, 핵심 로직은 패키지 안에 유지된다는 점이 중요하다.


In [ ]:
import pandas as pd


from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists() and (PROJECT_ROOT.parent / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(PROJECT_ROOT)

from src.config import get_paths
from src.ingestion import build_demo_index
from src.memory import (
    LongTermMemory,
    ShortTermMemory,
    VectorMemory,
    build_memory_augmented_context,
    display_memory,
    score_memory_importance,
    selective_memory_update,
    summarize_memory_events,
)
from src.utils import display_trace
from src.workflow import run_workflow_with_memory

pd.set_option('display.max_colwidth', 140)
paths = get_paths()
memory_store_path = paths.logs_dir / 'agent_memory_notebook_store.json'
if memory_store_path.exists():
    memory_store_path.unlink()


## 소개(Introduction)

agent가 memory를 필요로 하는 이유는 단순하다. 하나의 context window만으로는 모든 상호작용을 감당할 수 없기 때문이다. stateless pipeline은 현재 prompt와 검색된 문맥만으로 답한다. 반면 memory-enabled agent는 이전에 있었던 일, 학습된 사실, 사용자 선호 같은 정보를 다시 꺼내 쓸 수 있다.

context window 한계는 실무적으로 세 가지 문제를 만든다.

- 오래됐지만 중요한 정보가 범위 밖으로 밀려난다.
- 반복되는 사용자 선호를 계속 잊어버린다.
- 장기 작업이 turn 사이에서 끊긴다.

memory는 종류를 나눠 생각하는 것이 좋다.

- short-term memory: 최근 turn과 reasoning 상태를 담는 작업 버퍼
- long-term memory: 지속적으로 남겨야 하는 사실, 선호, 지식
- episodic memory: 특정 상호작용이나 사건의 기록
- semantic memory: 여러 경험에서 일반화된 사실과 개념

vector memory는 episodic memory나 semantic memory를 검색하는 메커니즘으로 자주 쓰인다. exact key 대신 similarity로 관련 memory를 찾는 방식이다.


## 구현

### 단기 메모리(Short-Term Memory)

short-term memory는 agent의 작업용 scratchpad에 가깝다. conversation buffer, 임시 reasoning state, 최근 tool output이 대표적인 예다. 아래 클래스는 이벤트를 순서대로 쌓고, 최근 `N`개만 다시 꺼내 보거나, 필요할 때 버퍼를 비울 수 있게 해준다.


In [ ]:
short_term = ShortTermMemory(max_items=6)
short_term.append_event('user', 'Can you explain the rollout timeline?')
short_term.append_event('assistant', 'The pilot runs from March 10, 2025 to April 4, 2025.')
short_term.append_event('user', 'Please keep future answers concise.')
short_term.append_event('assistant', 'Noted. I will keep answers concise when possible.')
short_term.to_frame()


이 작은 실험은 몇 개의 대화 turn을 쌓아두고, 그중 최근 memory만 다시 가져와 본다. 실제 agent가 전체 대화를 영원히 들고 가지 않고, working context만 유지하는 방식과 닮아 있다.


In [ ]:
recent_turns = short_term.last_n(3)
pd.DataFrame(recent_turns)


### 장기 메모리(Long-Term Memory)

long-term memory는 사용자 선호, 안정적인 사실, 지속적 프로젝트 지식처럼 오래 남아야 하는 정보를 담는다. 이 프로젝트에서는 JSON persistence를 사용해서, 저장 포맷 자체가 눈에 보이고 검증 가능하도록 만들었다.


In [ ]:
long_term = LongTermMemory(memory_store_path)
long_term.store_memory('mina_answer_style', 'Mina prefers concise answers that lead with exact dates.', category='preference')
long_term.store_memory('apollo_pilot_city', 'Team Apollo is piloting the scheduling template in Seoul.', category='fact')
reloaded_long_term = LongTermMemory(memory_store_path)
reloaded_long_term.to_frame()


장기 메모리는 한 셀이나 한 프로세스가 끝나도 다시 불러올 수 있어야 의미가 있다. 이 셀은 JSON 기반 저장소가 저장과 재로딩을 깔끔하게 round-trip하는지 확인한다.


In [ ]:
reloaded_long_term.retrieve_memory('mina_answer_style')


### 벡터 메모리(Vector Memory)

vector memory는 semantic retrieval을 담당한다. exact key를 묻는 대신, memory text를 벡터로 바꾼 뒤 similarity로 검색한다. 사용자가 같은 표현을 쓰지 않더라도, 이전 선호나 사건과 의미적으로 가까운 memory를 다시 찾고 싶을 때 유용하다.


In [ ]:
vector_memory = VectorMemory()
vector_memory.add_memory('Mina prefers concise rollout answers that begin with the exact launch date.', metadata={'kind': 'preference'}, importance=0.9)
vector_memory.add_memory('The rollout FAQ should mention the May 5, 2025 launch date.', metadata={'kind': 'fact'}, importance=0.8)
vector_memory.add_memory('The pilot retrospective is scheduled for June.', metadata={'kind': 'event'}, importance=0.5)
vector_search_results = vector_memory.search('How should I answer rollout timing for Mina?', top_k=3)
pd.DataFrame(vector_search_results)


## Agent에서의 메모리 검색(Memory Retrieval in Agents)

agent가 reasoning할 때는 retrieval이 둘로 나뉘는 경우가 많다.

1. 문서 같은 외부 근거를 검색한다.
2. 관련 memory 같은 내부 근거를 검색한다.

간단한 구조로 쓰면 `query -> retrieve docs -> retrieve memories -> combine context -> answer`다. 다음 셀은 기존 문서 retriever를 그대로 쓰면서, 그 위에 memory retrieval을 추가해본다.


In [ ]:
retriever = build_demo_index(persist=False)
retrieved_docs = retriever.search('When does the organization-wide rollout begin?', top_k=3)
context_bundle = build_memory_augmented_context(
    query='How should I answer rollout timing for Mina?',
    retrieved_docs=retrieved_docs,
    short_term_memory=short_term,
    long_term_memory=reloaded_long_term,
    vector_memory=vector_memory,
    top_k=3,
)
{
    'doc_sources': [doc['source'] for doc in context_bundle['retrieved_docs']],
    'memory_count': len(context_bundle['retrieved_memories']),
    'memory_texts': [memory['text'] for memory in context_bundle['retrieved_memories']],
}


### 메모리 업데이트 전략(Memory Update Strategy)

모든 것을 저장하는 것은 좋은 memory 정책이 아니다. noise가 늘어나고 retrieval이 느려지며, 정말 중요한 사실을 찾기 어려워진다. 그래서 append-only logging, recent history summarization, selective storage 같은 전략이 필요하다.

이 저장소는 rule-based importance score를 사용한다. 덕분에 update decision이 모델 안에 숨어버리지 않고 눈에 보이게 드러난다.


In [ ]:
candidates = [
    'Remember: Mina prefers concise rollout summaries with dates first.',
    'The user said thanks.',
    'Important: Team Apollo is piloting the scheduling template in Seoul.',
]
importance_frame = pd.DataFrame(
    {
        'candidate': candidates,
        'importance_score': [score_memory_importance(text, {'category': 'fact', 'source': 'user'}) for text in candidates],
    }
)
importance_frame


이 update 실험은 threshold를 넘는 memory만 저장한다. 핵심은 저장 정책을 inspectable하게 만드는 것이다. 어떤 항목은 남고 어떤 항목은 버려지는지 직접 보아야 memory 설계를 설명할 수 있다.


In [ ]:
update_decisions = [
    selective_memory_update(
        text='Remember: Mina prefers concise rollout summaries with dates first.',
        key='mina_style_rule',
        long_term_memory=reloaded_long_term,
        vector_memory=vector_memory,
        threshold=0.55,
        category='preference',
        metadata={'source': 'user'},
    ),
    selective_memory_update(
        text='The user said thanks.',
        key='low_signal_event',
        long_term_memory=reloaded_long_term,
        vector_memory=vector_memory,
        threshold=0.55,
        category='event',
        metadata={'source': 'user'},
    ),
]
pd.DataFrame(update_decisions)


## Agent workflow 안에서의 memory

memory는 전체 프로젝트를 갈아엎지 않고도 workflow에 연결할 수 있다. 이 저장소의 optional memory-aware runner는 memory를 검색하고 trace에 남기며, 답변 후 short-term이나 persistent store를 업데이트할 수 있다.

흐름으로 쓰면 `query -> retrieve docs -> retrieve memory -> generate answer -> update memory`다. 여기서 중요한 점은 memory 동작이 숨겨져 있지 않다는 것이다. memory가 답변을 어떻게 바꾸는지 trace로 확인할 수 있어야 한다.


In [ ]:
memory_state = run_workflow_with_memory(
    'When does the organization-wide rollout begin?',
    retriever=retriever,
    short_term_memory=short_term,
    long_term_memory=reloaded_long_term,
    vector_memory=vector_memory,
    include_memories_in_context=False,
    update_memory=True,
)
{
    'final_status': memory_state['final_status'],
    'final_answer': memory_state['final_answer'],
    'retrieved_memories': len(memory_state['retrieved_memories']),
    'memory_updates': len(memory_state['memory_updates']),
}


## 시각화(Visualization)

memory 디버깅이 중요한 이유는 hidden state가 조용히 쌓이기 쉽기 때문이다. 좋은 visualization은 short-term memory에 무엇이 있는지, long-term memory에 무엇이 저장되었는지, vector search 대상으로 무엇이 남아 있는지를 한 번에 보여줘야 한다.


In [ ]:
display_memory(
    short_term_memory=short_term,
    long_term_memory=reloaded_long_term,
    vector_memory=vector_memory,
)


이 trace view는 memory retrieval과 memory update가 workflow 어디에 들어오는지 보여준다. memory 동작이 "마법처럼" 숨어 있는 것이 아니라, 실행 흔적(trace) 안에 명확히 드러나는 구조라는 점이 중요하다.


In [ ]:
display_trace(memory_state['trace'])


## 실험

### 실험 1: memory가 답변 관련성을 높이는 경우

memory가 없으면 agent는 문서만 본다. memory가 있으면 사용자의 선호 답변 스타일이나, 현재 작업에 중요한 과거 사실까지 함께 끌어올 수 있다. 이 셀은 두 context bundle을 나란히 비교하게 해준다.


In [ ]:
query = 'How should I answer rollout timing for Mina?'
no_memory_bundle = build_memory_augmented_context(query=query, retrieved_docs=retrieved_docs, top_k=3)
with_memory_bundle = build_memory_augmented_context(
    query=query,
    retrieved_docs=retrieved_docs,
    short_term_memory=short_term,
    long_term_memory=reloaded_long_term,
    vector_memory=vector_memory,
    top_k=3,
)
pd.DataFrame(
    [
        {
            'mode': 'docs_only',
            'memory_count': len(no_memory_bundle['retrieved_memories']),
            'combined_context': str(no_memory_bundle['combined_context']),
        },
        {
            'mode': 'docs_plus_memory',
            'memory_count': len(with_memory_bundle['retrieved_memories']),
            'combined_context': str(with_memory_bundle['combined_context']),
        },
    ]
)


### 실험 2: memory pollution

memory는 많이 저장한다고 좋아지지 않는다. 이 셀은 의도적으로 noise가 되는 memory를 여러 개 넣은 뒤 similarity search를 다시 돌린다. 의미 없는 항목도 단어가 겹치면 ranking 상위로 섞일 수 있다는 점을 직접 볼 수 있다.


In [ ]:
polluted_memory = VectorMemory()
polluted_memory.add_memory('Mina prefers concise rollout answers that begin with the exact launch date.', metadata={'kind': 'preference'}, importance=0.9)
polluted_memory.add_memory('Mina is ordering snacks for the rollout celebration.', metadata={'kind': 'noise'}, importance=0.4)
polluted_memory.add_memory('Rollout posters should use the coral brand palette.', metadata={'kind': 'noise'}, importance=0.4)
polluted_memory.add_memory('The rollout FAQ should mention the May 5, 2025 launch date.', metadata={'kind': 'fact'}, importance=0.8)
pd.DataFrame(polluted_memory.search('How should I answer rollout timing for Mina?', top_k=4))


### 실험 3: memory summarization

summarization은 최근 기록을 모두 영구 저장하지 않고 압축하는 대표 전략이다. 여기서는 최근 short-term buffer를 하나의 summary string으로 접어서, 더 긴 주기의 저장소로 올릴 수 있는 형태를 만들어본다.


In [ ]:
summary_text = summarize_memory_events(short_term.events, limit=5)
reloaded_long_term.store_memory('recent_memory_summary', summary_text, category='summary')
pd.DataFrame(
    {
        'summary_text': [summary_text],
        'stored_summary': [reloaded_long_term.retrieve_memory('recent_memory_summary')['value']],
    }
)


## 결과 해석

이 노트북은 memory가 **선택적으로 저장되고, 검색 가능하며, traceable할 때** 도움이 된다는 점을 보여준다. 동시에 주요 리스크도 드러난다. noisy memory는 retrieval을 오염시키고, 숨겨진 update는 디버깅을 어렵게 만든다.

아래 표는 실험에서 확인한 핵심 관찰을 정리한 것이다.


In [ ]:
analysis_frame = pd.DataFrame(
    [
        {
            'theme': 'short_term_memory',
            'observation': f"buffer size after workflow run: {len(short_term.events)} events",
        },
        {
            'theme': 'long_term_memory',
            'observation': f"persisted items: {len(reloaded_long_term.list_items())}",
        },
        {
            'theme': 'vector_memory',
            'observation': f"top memory hit: {vector_search_results[0]['text'] if vector_search_results else 'none'}",
        },
        {
            'theme': 'workflow_integration',
            'observation': f"trace contains memory nodes: {any(entry['node'] == 'retrieve_memories' for entry in memory_state['trace'])}",
        },
    ]
)
analysis_frame


## 핵심 정리

- 이 실험을 통해 memory는 continuity, personalization, durable facts를 유지하는 데 큰 도움이 된다는 점을 확인했다.
- short-term memory는 작업 맥락 유지에, long-term과 vector memory는 지속성과 retrieval에 강점이 있다.
- selective storage 없이 memory를 쌓으면 relevance가 오히려 나빠질 수 있다.
- 면접에서는 memory를 이야기할 때 **무엇을 저장하는가, 어떻게 검색하는가, 어떻게 업데이트를 통제하는가** 세 축을 강조하면 좋다.
- 더 나아간 개선 포인트로는 summarization 고도화, recency-aware ranking, preference용 저장소와 factual memory 저장소 분리가 있다.
